# Task 2 — Kaggle launcher

Thin launcher only: all logic is in `task2/` and `kaggle/task2_run.py`.

Settings: **GPU T4 x2** (Task 2 uses GPU 0), **Internet on**, secret **HF_TOKEN**. After any interruption run all cells again: splits, feature shards and training checkpoints are pulled from the HF repo and finished work is skipped.

In [ ]:
REPO_URL= "https://github.com/Zeeba1705/atml_PA3.git"
BRANCH= "task2-vlm"
SYNC_REPO= "zeeba1705/atml-pa3"
#12 h session limit minus setup time
MAX_HOURS= 11.0

In [ ]:
import os
import subprocess
import sys
import time

if not os.path.exists("/kaggle/working/atml_PA3"):
    !git clone -b {BRANCH} {REPO_URL} /kaggle/working/atml_PA3

%cd /kaggle/working/atml_PA3
!git fetch -q && git checkout -q {BRANCH} && git pull -q
!git log --oneline -1

In [ ]:
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

import torch
#stop here if the notebook has no GPU (Settings -> Accelerator -> GPU T4 x2)
assert torch.cuda.is_available(), "No GPU: set Accelerator to GPU T4 x2 and run again"

In [ ]:
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"]= UserSecretsClient().get_secret("HF_TOKEN")

### Pipeline on GPU 0: splits → SigLIP features → 3-stage training
Log in `logs/task2.log`, copied here every 2 min.

In [ ]:
os.makedirs("logs", exist_ok=True)
os.environ["PYTHONUNBUFFERED"]= "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"]= "1"
os.environ["CUDA_VISIBLE_DEVICES"]= "0"

log= open("logs/task2.log", "a")
proc= subprocess.Popen(
    [
        sys.executable, "kaggle/task2_run.py",
        "--sync_repo", SYNC_REPO,
        "--max_hours", str(MAX_HOURS)
    ],
    stdout=log,
    stderr=subprocess.STDOUT
)

#keep the notebook alive (needed for Save & Run All) and copy new log lines every 2 min
seen= 0
while True:
    running= proc.poll() is None
    time.sleep(120 if running else 0)

    with open("logs/task2.log") as f:
        f.seek(seen)
        new= f.read()
        seen= f.tell()
    for line in new.splitlines():
        print(line)

    if not running:
        break

print("exit code", proc.returncode)